# Production RAG Pipeline — Colab, Self-Contained (`gemma-4-26b-a4b-it` via Modal)

Self-contained Colab version of `production_pipeline_demo.ipynb`, validating the
**same winning retrieval recipe** from the original ablation sweep — dense-only
FAISS retrieval, `sentence-transformers/all-MiniLM-L6-v2`, per-dataset fixed-word
chunking, `top_k=5`, no reranking/fusion/query-transform — across all three
customer-support subsets (`techqa` / `emanual` / `delucionqa`) of
`galileo-ai/ragbench`, but with a single new generator+judge model:
**`gemma-4-26b-a4b-it`**, served from your own Modal endpoint.

**What's different from `production_pipeline_demo.ipynb`:** that notebook loads
its per-dataset RAG config from `experiment_configs/{dataset}_production_experiment.yaml`
*plus* `rag-experiments/{dataset}-production/config/*.yaml` — two separate files
you'd have to hand-edit to change the model/provider. Here, **section 5 below is
the only config** — one Python dict per dataset — and this notebook writes the
YAML the framework expects at runtime. Nothing to open or edit outside this file.

**What's reused unchanged:** the actual `rag_cust_support` framework code
(chunking/embedding/FAISS/TRACe-judge implementations) — this notebook mounts
your Drive copy of the repo and imports it, exactly like the original notebook
does, so none of that logic is reimplemented or duplicated here.

**Provider plumbing:** `gemma-4-26b-a4b-it` is not a new provider class — the
existing `openrouter` provider is already a generic OpenAI-compatible client
(reads `base_url` from its config), so it's just registered under the name
`modal` pointed at your Modal endpoint. Same model/provider is used for both
generation and the TRACe judge (no fallback provider — if Modal is down, the
run fails loudly rather than silently switching models).

**Isolated output dirs** (`rag-experiments/{dataset}-production-colab-gemma/`,
`cache_production_colab_gemma_{dataset}/`) so this never collides with the
existing OpenRouter/Llama production run's history or cache.

## 1. Setup — mount Drive & install dependencies

Mounting Drive (not `git clone`) because `rag_cust_support` is a private GitHub
repo — this matches how `production_pipeline_demo.ipynb` already does it, so no
extra GitHub-token secret is needed. If you'd rather clone directly, swap this
cell for `git clone https://<token>@github.com/bhupendra-bhoi/rag_cust_support.git`
using a token stored in Colab Secrets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

get_ipython().system('pip install -q datasets faiss-cpu sentence-transformers torch groq openai pyyaml pandas rank_bm25 nltk')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Point at the repo + imports

In [ ]:
import sys
import os
import yaml
from pathlib import Path

# --- Point this at wherever the rag_cust_support repo lives in YOUR Drive. ---
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone/rag_cust_support')
assert PROJECT_ROOT.exists(), f'{PROJECT_ROOT} not found — update this path to your Drive copy of the repo.'

os.chdir(PROJECT_ROOT)
project_root = PROJECT_ROOT
# Make THIS repo win on sys.path (avoids importing a stale rag-foundry copy).
sys.path = [p for p in sys.path if 'rag-foundry' not in p]
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner
from providers.config import ProviderConfig
from providers.provider_manager import ProviderManager
import experiment.experiment_runner as _er

print('Current directory:', Path.cwd())
print('experiment_runner loaded from:', _er.__file__)
assert 'rag-foundry' not in _er.__file__, 'Still importing the old rag-foundry code! Restart runtime.'

Current directory: /content/drive/MyDrive/Capstone/rag_cust_support
experiment_runner loaded from: /content/drive/MyDrive/Capstone/rag_cust_support/experiment/experiment_runner.py


## 3. Secrets — Modal endpoint for `gemma-4-26b-a4b-it`

Add both to the Colab **Secrets** tab (key icon in the left sidebar) and grant
this notebook access: `MODAL_BASE_URL` (your deployed endpoint's OpenAI-compatible
base URL, e.g. `https://<workspace>--<app-name>.modal.run/v1`) and
`MODAL_PROXY_KEY` (your Modal proxy key).

In [ ]:
from google.colab import userdata

MODAL_BASE_URL = userdata.get('MODAL_BASE_URL')
os.environ['MODAL_PROXY_KEY'] = userdata.get('MODAL_PROXY_KEY')

print('MODAL_BASE_URL loaded:', bool(MODAL_BASE_URL))
print('MODAL_PROXY_KEY loaded:', bool(os.environ.get('MODAL_PROXY_KEY')))

MODAL_BASE_URL loaded: True
MODAL_PROXY_KEY loaded: True


## 4. Sanity check — confirm the endpoint responds before spending on the full run

Registers the `modal` provider and sends one trivial no-context query. Catches a
wrong `MODAL_BASE_URL`/`MODAL_PROXY_KEY` or a cold/misconfigured endpoint here,
instead of partway through a long checkpointed pipeline run.

In [ ]:
MODEL_NAME = 'gemma-4-26b-a4b-it'  # <-- exact model id your Modal deployment expects

_sanity_provider_config = ProviderConfig(
    type='openai',
    api_key_env='MODAL_PROXY_KEY',
    params={'base_url': MODAL_BASE_URL, 'cooldown_seconds': 60},
)
ProviderManager.register('modal', _sanity_provider_config.type, _sanity_provider_config)

_resp = ProviderManager.get_provider('modal').generate(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': 'Reply with just the word OK.'}],
    temperature=0.0, max_tokens=10,
)
_text = _resp.choices[0].message.content.strip()
print('[OK] modal endpoint responded:', repr(_text)) if _text else print('[EMPTY] modal endpoint returned no text — check MODAL_BASE_URL/MODEL_NAME')

Using key #0: ****kv8w
[OK] modal endpoint responded: 'OK'


## 5. The ONE config cell — per-dataset chunking + prompts

This replaces both external YAML files from the original notebook. Everything
you might want to tweak (chunk size, prompts, token budgets, sample count) lives
right here — edit this cell, not a file elsewhere. Retrieval settings (dense-only,
`all-MiniLM-L6-v2`, `top_k=5`) are the winning config from the original ablation
sweep and are intentionally the same for all three datasets; only chunk size and
the generation prompt are dataset-specific, exactly as in `production/configs/*.yaml`.

In [ ]:
END_INDEX = 20  # rows per dataset; raise up to each dataset's full test-split size
                # once you're ready — runs are checkpoint-resumable, so raising
                # this later re-costs nothing already completed.

DATASET_SETTINGS = {
    'techqa': {
        'limit': 314,
        'max_words': 200, 'overlap_words': 20,
        'max_tokens': 750,
        'system_prompt': (
            'You are an IBM technical-support question answering assistant.\n'
            'Your task is to answer questions using ONLY information from the retrieved\n'
            'technote passages.\n\n'
            'CRITICAL RULES:\n'
            '1. Answer ONLY from the passages provided. Do not use external knowledge.\n'
            '2. Include all relevant steps, error codes, product versions, APAR numbers,\n'
            '   and configuration details found in the passages.\n'
            '3. Preserve exact technical identifiers (version strings, file names,\n'
            '   parameter names, error messages) verbatim.\n'
            '4. Do NOT add phrases like "based on general knowledge" or hedge unnecessarily.\n'
            '5. If the passages do not contain enough information, respond with exactly:\n'
            '   "The passages do not provide sufficient information to answer this question."\n'
            '6. Every claim must be directly supported by the passages provided.'
        ),
        'user_prompt': (
            'Passages:\n{context}\n\n'
            'Question: {query}\n\n'
            'Answer (from passages only, include all relevant steps and technical details):'
        ),
    },
    'emanual': {
        'limit': 132,
        'max_words': 128, 'overlap_words': 20,
        'max_tokens': 512,
        'system_prompt': (
            'You are a consumer-electronics product-support assistant. Answer questions\n'
            'about device features and settings using ONLY the provided user-manual passages.\n\n'
            'CRITICAL RULES:\n'
            '1. Answer ONLY from the passages provided. Do not use outside knowledge.\n'
            '2. Give exact, step-by-step instructions. Preserve menu paths, button names,\n'
            '   setting names, and on-screen labels verbatim\n'
            '   (e.g. "Settings > Support > Self Diagnosis > Signal Information").\n'
            '3. Be concise and procedural - give the steps, not background.\n'
            '4. If the passages do not contain the answer, respond with exactly:\n'
            '   "The passages do not provide sufficient information to answer this question."\n'
            '5. Every step must be directly supported by the passages provided.'
        ),
        'user_prompt': (
            'Passages:\n{context}\n\n'
            'Question: {query}\n\n'
            'Answer (from passages only, give exact steps and menu paths):'
        ),
    },
    'delucionqa': {
        'limit': 184,
        'max_words': 160, 'overlap_words': 24,
        'max_tokens': 512,
        'system_prompt': (
            'You are an automotive owner\'s-manual support assistant. Answer questions\n'
            'about vehicle features, controls, and procedures using ONLY the provided\n'
            'owner\'s-manual passages.\n\n'
            'CRITICAL RULES:\n'
            '1. Answer ONLY from the passages provided. Do not use outside knowledge.\n'
            '2. Give exact, step-by-step procedures. Preserve exact control names, button\n'
            '   names, warning/CAUTION text, and feature labels verbatim.\n'
            '3. Include any safety warnings (WARNING/CAUTION) that the passages attach to\n'
            '   the procedure.\n'
            '4. Be concise and procedural - give the steps, not background.\n'
            '5. If the passages do not contain the answer, respond with exactly:\n'
            '   "The passages do not provide sufficient information to answer this question."\n'
            '6. Every claim must be directly supported by the passages provided.'
        ),
        'user_prompt': (
            'Passages:\n{context}\n\n'
            'Question: {query}\n\n'
            'Answer (from passages only, give exact steps, control names, and any safety warnings):'
        ),
    },
}

def build_rag_config(dataset):
    """Inner RAGConfig dict — dense-only retrieval + gemma-4-26b-a4b-it via Modal."""
    s = DATASET_SETTINGS[dataset]
    return {
        'mode': 'dev',
        'name': f'{dataset}_production_colab_gemma',
        'cache': {'enabled': True, 'cache_dir': f'./cache_production_colab_gemma_{dataset}'},
        'providers': {
            'modal': {
                'type': 'openai',
                'api_key_env': 'MODAL_PROXY_KEY',
                'params': {'base_url': MODAL_BASE_URL, 'cooldown_seconds': 60},
            },
        },
        'chunking': {
            'type': 'fixed_word',
            'config': {'max_words': s['max_words'], 'overlap_words': s['overlap_words']},
        },
        'embedding': {
            'type': 'sentence_transformer',
            'config': {'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'dimension': 384},
        },
        'vector_store': {'type': 'faiss', 'config': {'dimension': 384}},
        'retrieval': {'search': {'searches': [{'type': 'dense', 'config': {'top_k': 5}}]}},
        'generation': {
            'strategy': 'default',
            'provider': 'modal',
            'config': {
                'model': MODEL_NAME,
                'temperature': 0.0,
                'max_tokens': s['max_tokens'],
                'system_prompt': s['system_prompt'],
                'user_prompt': s['user_prompt'],
            },
        },
        'evaluation': {
            'type': 'trace',
            'provider': 'modal',
            'config': {'model': MODEL_NAME, 'temperature': 0.0, 'max_tokens': 2000},
        },
    }

## 6. Build isolated experiment dirs + run

For each dataset: write the config from section 5 into a fresh, isolated
`config/` dir (this is the step that used to require hand-editing a YAML file —
here it's generated fresh from the dict above every run), then run the same
`ExperimentRunner` flow as the original notebook (load data → run pipeline →
evaluate with the TRACe judge → generate report).

In [ ]:
DATASETS = ['techqa', 'emanual', 'delucionqa']

runners = {}
reports = {}

for dataset in DATASETS:
    print(f'\n{"="*80}\n{dataset.upper()} — production config (Colab, gemma-4-26b-a4b-it)\n{"="*80}')

    exp_root = project_root / 'rag-experiments' / f'{dataset}-production-colab-gemma'
    config_dir = exp_root / 'config'
    config_dir.mkdir(parents=True, exist_ok=True)
    # Fresh config every run — this is the file that replaces manual YAML editing.
    with open(config_dir / f'{dataset}_production_colab_gemma.yaml', 'w') as f:
        yaml.safe_dump(build_rag_config(dataset), f, sort_keys=False)

    experiment_config = ExperimentConfig(
        config_dir=config_dir,
        report_dir=exp_root / 'reports',
        temp_dir=exp_root / 'temp',
        start_index=0,
        end_index=END_INDEX,
        cache={'enabled': True, 'cache_dir': f'./cache_production_colab_gemma_{dataset}'},
        data_loader={
            'type': 'huggingface',
            'config': {
                'dataset_name': 'galileo-ai/ragbench',
                'subset': dataset,
                'split': 'test',
                'limit': DATASET_SETTINGS[dataset]['limit'],
            },
        },
        data_parser='noop',
        evaluation={
            'type': 'trace',
            'provider': 'modal',
            'config': {'model': MODEL_NAME, 'temperature': 0.0, 'max_tokens': 2000},
        },
    )
    runner = ExperimentRunner(experiment_config)
    runners[dataset] = runner

    documents, raw_data = runner.load_data()
    print(f'Loaded {len(raw_data)} raw rows -> {len(documents)} parsed documents')

    configs = runner.load_configs()
    assert len(configs) == 1, f'Expected exactly 1 config for {dataset}, found {len(configs)}'
    print(f'Running config: {configs[0].name}')

    runs = runner.run(documents, raw_data)
    runs = runner.evaluate_runs(runs)
    reports[dataset] = runner.generate_reports(runs)


TECHQA — production config (Colab, gemma-4-26b-a4b-it)
Loading HuggingFace dataset: galileo-ai/ragbench/techqa (test)...
Loaded 314 samples
Loaded 314 raw rows -> 1570 parsed documents
Running config: techqa_production_colab_gemma
Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 0sUsing key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 3sUsing key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
Progress: 5/20 (25.0%) | QPS: 1.25 | ETA: 12s | Elapsed: 4sUsing key #0: ****kv8w
Progress: 6/20 (30.0%) | QPS: 1.00 | ETA: 14s | Elapsed: 6sUsing key #0: ****kv8w
Progress: 7/20 (35.0%) | QPS: 0.87 | ETA: 15s | Elapsed: 8sUsing key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
Progress: 12/20 (60.0%) | QPS: 1.33 | ETA: 6s | Elapsed: 9sUsing key #0: ****kv8w
Using key #0: ***

Using key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
Using key #0: ****kv8w
[delucionqa_production_colab_gemma] Evaluation complete → /content/drive/MyDrive/Capstone/rag_cust_support/rag-experiments/delucionqa-production-colab-gemma/temp/delucionqa_production_colab_gemma.jsonl


In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

for dataset in DATASETS:
    print(f'\n{"="*80}\n{dataset.upper()}\n{"="*80}')
    print(f'Raw reports for {dataset}: {reports.get(dataset, "No report found")}')
    try:
        comparison = runners[dataset].compare()
        display(comparison.to_dataframe())
    except Exception as e:
        print(f'Error generating comparison for {dataset}: {e}')



TECHQA
Raw reports for techqa: [Report(title='RAG Multi-Config Evaluation Report', strategy_name='detailed_query', sections=[ReportSection(config_name='techqa_production_colab_gemma', per_query=                                                                              query  \
0   Using cobol copybooks Sometimes, there will be errors/fields missing in type...   
1   Is WebSphere Transformation Extender (WTX) supported for IBM Integration Bus...   
2   Want to find out if Microsoft Edge is supported with ICC? Want to find out M...   
3   What IBM Business Process Manager version is affected by the Apache Commons ...   
4   Error while installing SpSS modeler trial version I have downloaded the SPSS...   
5   Where can I find more details about WMB Fix Pack 8.0.0.5? Where can find mor...   
6   How to resolve the StaleConnectionException in WebSphere Application Server?...   
7   How do I resolve a WSVR0605W hung thread issue in the WebSphere MQ Resource ...   
8   I need to transfer

,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,techqa_production_colab_gemma,0.2567,0.2479,0.1472,0.1525,0.2938,0.5803,1.0,0.45



EMANUAL
Raw reports for emanual: [Report(title='RAG Multi-Config Evaluation Report', strategy_name='detailed_query', sections=[ReportSection(config_name='emanual_production_colab_gemma', per_query=                                                                              query  \
0                            I want to  enter into Ambient mode. How can I do that?   
1                                              Where do I find signal information ?   
2   How can I view the channels that are serached by auto program function and H...   
3                                                             Can I configure Tint?   
4                                      How do I fix the missing/wrong color issue ?   
5                                        How do I fix blurring issues on TV screen?   
6                                               What is the use of universal guide?   
7                                               What is the feature of Bixby guide?   
8                  

,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,emanual_production_colab_gemma,0.3008,0.1813,0.1584,0.1377,0.6191,0.301,0.9,0.2



DELUCIONQA
Raw reports for delucionqa: [Report(title='RAG Multi-Config Evaluation Report', strategy_name='detailed_query', sections=[ReportSection(config_name='delucionqa_production_colab_gemma', per_query=                                                                       query  \
0                             What if I fail to latch the tailgate properly?   
1                  What kind of safety features are implemented in this car?   
2                                  When will the Automatic SOS be triggered?   
3                   What happens if I accidentally push the SOS Call button?   
4                                                           What is the DEF?   
5                  What may cause erratic or noisy performance of the radio?   
6                                 how to calculate the gross trailer weight?   
7                                              What can the ASIST button do?   
8                                      What does the Door Off Mirror Kit 

,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,delucionqa_production_colab_gemma,0.2679,0.1576,0.2111,0.1112,0.7845,0.177,0.9474,0.0526


## 7. Results

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

for dataset in DATASETS:
    print(f'\n{"="*80}\n{dataset.upper()}\n{"="*80}')
    comparison = runners[dataset].compare()
    display(comparison.to_dataframe())


TECHQA


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,techqa_production_colab_gemma,0.2567,0.2479,0.1472,0.1525,0.2938,0.5803,1.0,0.45



EMANUAL


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,emanual_production_colab_gemma,0.3008,0.1813,0.1584,0.1377,0.6191,0.301,0.9,0.2



DELUCIONQA


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,delucionqa_production_colab_gemma,0.2679,0.1576,0.2111,0.1112,0.7845,0.177,0.9474,0.0526


## 8. Conclusion

This validates `gemma-4-26b-a4b-it` (via Modal) as generator + judge on the
customer-support domain's already-chosen retrieval architecture (dense-only,
`all-MiniLM-L6-v2`, `top_k=5`, per-dataset fixed-word chunking) — the retrieval
side is unchanged from `production/configs/*.yaml`; only the model/provider
changed, to compare directly against the `meta-llama/llama-3.3-70b-instruct`
(OpenRouter) baseline in the original `production_pipeline_demo.ipynb`.

**Known open limitation, carried over from the original sweep, not specific to
this model**: techqa's real (non-refusal) adherence stayed near-zero regardless
of retrieval strategy in the original sweep — worth checking whether that
persists with this generator too, since it looked like a generation-prompt/
domain issue rather than a retrieval problem.

To swap models/providers again later, edit `MODEL_NAME`/`MODAL_BASE_URL` and the
`providers` block in section 5 — no other file in this notebook or the repo
needs to change.